<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# <a id='toc1_'></a>[Build Netflix-like recommendation systems with Sklearn](#toc0_)


Estimated time needed: **30** minutes


In this guided project, we explore an easy way of building recommendation systems. We begin by looking at popular-based recommendations and then ease into a simple application of how content-based and collaborative filtering recommendation works.


**Table of contents**<a id='toc0_'></a>    
 
  - [Background on recommendation systems](#toc1_1_)    
    - [Types of recommendation systems](#toc1_1_1_)    
  - [Objectives](#toc1_2_)    
  - [Setup](#toc1_3_)    
    - [Installing required libraries](#toc1_3_1_)    
    - [Importing required libraries](#toc1_3_2_)    
  - [Exploratory data analysis (EDA)](#toc1_4_)    
  - [Popularity-based recommendation](#toc1_5_)    
    - [Exercise 1 - Get the top 5 suggestions sorting by score in descending order](#toc1_5_1_)    
  - [Content-based recommendation](#toc1_6_)    
    - [Exercise 2 - Check the recommendations for the movie 'Toy Story 2 (1999)'](#toc1_6_1_)    
  - [Collaborative filtering](#toc1_7_)    
    - [Exercise 3 - Check the recommendations for the movie 'Jurassic Park (1993)'](#toc1_7_1_)   


## <a id='toc1_1_'></a>[Background on recommendation systems](#toc0_)

Recommendation systems have become an integral part of our digital lives, subtly shaping the content we consume and the products we buy. From suggesting movies on Netflix to recommending products on Amazon, these systems help users navigate vast amounts of information by providing personalized suggestions based on their preferences and behaviors.

### <a id='toc1_1_1_'></a>[Types of recommendation systems](#toc0_)

There are several types of recommendation systems, each with its unique approach to generating recommendations:

1. **Popularity-based recommendation**: Popular-based recommendation systems are straightforward to implement because they don’t require complex algorithms or user-specific data. They often rely on basic statistics like item frequency and offer the same suggestions to all users, focusing on what is popular among the majority.

2. **Content-based filtering**: This approach focuses on the characteristics of the items themselves. It recommends items that are similar to those the user has shown interest in, based on item features.

3. **Collaborative filtering**: This method relies on the collective preferences of users. It can be user-based, where recommendations are made based on the preferences of similar users, or item-based, where recommendations are made based on items that are similar to what the user has liked in the past.


## <a id='toc1_2_'></a>[Objectives](#toc0_)



After completing this lab, you are able to:



- Understand the basic concepts and types of recommendation systems.

- Implement a simple popularity-based recommendation system.

- Implement a content-based recommendation system.

- Implement a item-based recommendation system.



----


## <a id='toc1_3_'></a>[Setup](#toc0_)

For this lab, you use the following libraries:

*   [`pandas`](https://pandas.pydata.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for managing the data.
*   [`sklearn`](https://scikit-learn.org/stable/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for machine learning and machine-learning-pipeline related functions.


### <a id='toc1_3_1_'></a>[Installing required libraries](#toc0_)


In [1]:
%pip install tqdm==4.66.4  | tail -n 1
%pip install pandas==2.1.4  | tail -n 1
%pip install scikit-learn==1.5.1  | tail -n 1

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### <a id='toc1_3_2_'></a>[Importing required libraries](#toc0_)


In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
import statistics


# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass

import warnings

warnings.warn = warn
warnings.filterwarnings('ignore')


The dataset is taken from [Kaggle](https://www.kaggle.com/datasets/shubhammehta21/movie-lens-small-latest-dataset/data).
This dataset describes 5-star rating and free-text tagging activity from MovieLens, a movie recommendation service. Users were selected at random for inclusion. No demographic information is included. Each user is represented by an ID, and no other information is provided.

The data are contained in the files movies.csv, ratings.csv and tags.csv. 

In the `movies.csv` file:
- `movieId`: ID of the movie/show (unique)
- `title`: Title of the movie/show
- `genres`: Genre of the show
  
In the `ratings.csv` file:
- `userId`: ID of the user who gave a rating
- `movieId`: ID of the movie/show rated
- `rating`: Rating given to the show
- `timestamp`: Time when the rating was specified
  
In the `tags.csv` file:
- `userId`: ID of the user who gave a rating
- `movieId`: ID of the movie/show rated
- `tag`: Tags given to the show
- `timestamp`: Time when the rating was specified

Now, let's load these datasets into a pandas DataFrame.



In [3]:
movie_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/BxZuF3FrO7Bdw6McwsBaBw/movies.csv')
rating_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/R-bYYyyf7s3IUE5rsssmMw/ratings.csv')
tag_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/UZKHhXSl7Ft7t9mfUFZJPQ/tags.csv')

Let's look at some samples rows from the dataset we loaded:


In [4]:
movie_df.sample(5)

,movieId,title,genres
1247,1658,"Life Less Ordinary, A (1997)",Romance|Thriller
5661,27592,Sympathy for Mr. Vengeance (Boksuneun naui geo...,Crime|Drama
6638,56367,Juno (2007),Comedy|Drama|Romance
6023,38388,Goal! The Dream Begins (Goal!) (2005),Drama
1218,1617,L.A. Confidential (1997),Crime|Film-Noir|Mystery|Thriller


In [5]:
tag_df.sample(5)

,userId,movieId,tag,timestamp
1627,474,2424,remake,1137202717
1086,474,339,coma,1138137781
2097,474,6244,In Netflix queue,1137201921
1010,474,40,South Africa,1137202107
2668,477,60069,last man on earth,1241396252


In [6]:
rating_df.sample(5)

,userId,movieId,rating,timestamp
22642,155,333,4.0,961861110
23447,160,1997,5.0,986319784
9910,64,8874,5.0,1161528841
38425,263,7458,3.0,1090948081
47635,307,52458,2.5,1186878639


In [7]:
print(movie_df.shape)
print(rating_df.shape)
print(tag_df.shape)

(9742, 3)
(100836, 4)
(3683, 4)


In [8]:
# We will merge the three dataframes to create a single dataframe that contains all the information we need.
user_movie_df = movie_df.merge(rating_df, on = 'movieId', how = 'inner')

# let's see this shape first
user_movie_df.shape

(100836, 6)

In [9]:
user_movie_df.head()

,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1,4.0,964982703
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5,4.0,847434962
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,7,4.5,1106635946
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,15,2.5,1510577970
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,17,4.5,1305696483


In [10]:
df = user_movie_df.merge(tag_df, on = ['movieId', 'userId'], how = 'inner')
df.shape

(3476, 8)

In [11]:
df.head()

,movieId,title,genres,userId,rating,timestamp_x,tag,timestamp_y
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,336,4.0,1122227329,pixar,1139045764
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,474,4.0,978575760,pixar,1137206825
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,567,3.5,1525286001,fun,1525286013
3,2,Jumanji (1995),Adventure|Children|Fantasy,62,4.0,1528843890,fantasy,1528843929
4,2,Jumanji (1995),Adventure|Children|Fantasy,62,4.0,1528843890,magic board game,1528843932


In [12]:
# Here, we will drop the timestamp columns as they are not needed for our analysis.
df.drop(columns = ['timestamp_x', 'timestamp_y'], inplace = True)
df

,movieId,title,genres,userId,rating,tag
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,336,4.0,pixar
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,474,4.0,pixar
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,567,3.5,fun
3,2,Jumanji (1995),Adventure|Children|Fantasy,62,4.0,fantasy
4,2,Jumanji (1995),Adventure|Children|Fantasy,62,4.0,magic board game
...,...,...,...,...,...,...
3471,187595,Solo: A Star Wars Story (2018),Action|Adventure|Children|Sci-Fi,62,4.0,star wars
3472,193565,Gintama: The Movie (2010),Action|Animation|Comedy|Sci-Fi,184,3.5,anime
3473,193565,Gintama: The Movie (2010),Action|Animation|Comedy|Sci-Fi,184,3.5,comedy
3474,193565,Gintama: The Movie (2010),Action|Animation|Comedy|Sci-Fi,184,3.5,gintama


---
## <a id='toc1_4_'></a>[Exploratory data analysis (EDA)](#toc0_)


Before doing any preprocessing, we will be performing some simple exploratory data analysis (EDA) to know about our dataset. This includes looking at the number of unique values/number of duplicate values, the distributions, etc.

First, looking at the shape of the `pd.DataFrame`


In [13]:
print('Number of rows: ' , df.shape[0])
print('Number of columns: ' , df.shape[1])

Number of rows:  3476
Number of columns:  6


Looking at the data type of each columns:


In [14]:
df.dtypes

movieId      int64
title       object
genres      object
userId       int64
rating     float64
tag         object
dtype: object

Next, let's see if we have any null values:


In [15]:
# Deal with null values
df.isnull().any()

movieId    False
title      False
genres     False
userId     False
rating     False
tag        False
dtype: bool

## <a id='toc1_5_'></a>[Popularity-based recommendation](#toc0_)

The popularity based recommendation recommends items, in this case, movies, based on what is popular accross the site. It is the most basic recommendation system. The system identifies popular items by considering metrics such as the number of views, ratings, or purchases and suggests these items to all users. For this type of recommendation system, all users get the same recommendations. The system can suggest items based on what's popular in your country. 

This approach ensures that users are aware of current popular content, which can be useful for new users who have not yet developed a viewing history on the platform. However, this is also a limitation because everyone receives the same suggestions, which may not always be relevant or interesting to them. This lack of specificity can result in a less engaging user experience compared to more personalized recommendation systems.


In [16]:
df_1 = df
df_1.head()

,movieId,title,genres,userId,rating,tag
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,336,4.0,pixar
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,474,4.0,pixar
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,567,3.5,fun
3,2,Jumanji (1995),Adventure|Children|Fantasy,62,4.0,fantasy
4,2,Jumanji (1995),Adventure|Children|Fantasy,62,4.0,magic board game


Next, we will be calculating the number of votes and the average rating for each movie.


In [17]:
num_votes = df_1.groupby('movieId').size().reset_index(name='numVotes')
num_votes

,movieId,numVotes
0,1,3
1,2,4
2,3,2
3,5,2
4,7,1
...,...,...
1459,183611,3
1460,184471,3
1461,187593,3
1462,187595,2


In [18]:
# Merge the numVotes back into the original DataFrame
df_1 = pd.merge(df_1, num_votes, on='movieId')

df_1.head()

,movieId,title,genres,userId,rating,tag,numVotes
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,336,4.0,pixar,3
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,474,4.0,pixar,3
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,567,3.5,fun,3
3,2,Jumanji (1995),Adventure|Children|Fantasy,62,4.0,fantasy,4
4,2,Jumanji (1995),Adventure|Children|Fantasy,62,4.0,magic board game,4


In [19]:
avg_ratings = df_1.groupby('movieId')['rating'].mean().reset_index(name='avgRating')
avg_ratings

,movieId,avgRating
0,1,3.833333
1,2,3.750000
2,3,2.500000
3,5,1.500000
4,7,3.000000
...,...,...
1459,183611,4.000000
1460,184471,3.500000
1461,187593,4.000000
1462,187595,4.000000


In [20]:
# Merge the avgRating back into the original DataFrame
df_1 = pd.merge(df_1, avg_ratings, on='movieId')

In [21]:
df_1.sample(6)

,movieId,title,genres,userId,rating,tag,numVotes,avgRating
708,1079,"Fish Called Wanda, A (1988)",Comedy|Crime,474,4.5,fish,1,4.50
736,1101,Top Gun (1986),Action|Romance,474,3.5,Navy,2,2.75
424,410,Addams Family Values (1993),Children|Comedy|Fantasy,62,4.5,Christopher Lloyd,6,4.50
2078,6541,"League of Extraordinary Gentlemen, The (a.k.a....",Action|Fantasy|Sci-Fi,62,3.5,Peta Wilson,6,3.50
1442,3135,"Great Santini, The (1979)",Drama,474,4.0,fatherhood,1,4.00
3230,114627,Angel's Egg (Tenshi no tamago) (1985),Animation|Drama|Fantasy,567,3.5,atmospheric,4,3.50


In [22]:
df_1.drop_duplicates(subset = ['movieId', 'title', 'avgRating', 'numVotes'], inplace = True)
df_1

,movieId,title,genres,userId,rating,tag,numVotes,avgRating
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,336,4.0,pixar,3,3.833333
3,2,Jumanji (1995),Adventure|Children|Fantasy,62,4.0,fantasy,4,3.750000
7,3,Grumpier Old Men (1995),Comedy|Romance,289,2.5,moldy,2,2.500000
9,5,Father of the Bride Part II (1995),Comedy,474,1.5,pregnancy,2,1.500000
11,7,Sabrina (1995),Comedy|Romance,474,3.0,remake,1,3.000000
...,...,...,...,...,...,...,...,...
3461,183611,Game Night (2018),Action|Comedy|Crime|Horror,62,4.0,Comedy,3,4.000000
3464,184471,Tomb Raider (2018),Action|Adventure|Fantasy,62,3.5,adventure,3,3.500000
3467,187593,Deadpool 2 (2018),Action|Comedy|Sci-Fi,62,4.0,Josh Brolin,3,4.000000
3470,187595,Solo: A Star Wars Story (2018),Action|Adventure|Children|Sci-Fi,62,4.0,Emilia Clarke,2,4.000000


We will be calculating the weighted score for each type. Usually, we would think that a good score results when the rating is high and the number of votes is also high. For instance, suppose you were browsing to choose a restaurant to dine at on your trip. If restaurant A had score 8.5 with 100,000 votes and restaurant B had score 8.5 but with 10 votes, we would be more convinced that restaurant A is more enjoyable and popular. Similarly, if restaurant C had score 5.0 with 1000 votes and restaurant D had score 5.0 with 1 vote, we may not automatically think that restaurant D was not enjoyable (but we do know that it is not popular), since only one person submitted a rating, if another person gave it score 10, this would immediately bump the score of restaurant D to 7.5.

The code below creates a new column `df['score']` that calculates the weighted average score for each movie.


In [23]:
import statistics

# The formula below is often referred to as the Bayesian Average

# Define the function to calculate the weighted score
def calculate_weighted_score(avgRating, num_votes, C, m):
    return (num_votes * avgRating + m * C) / (num_votes + m)

# Calculate the global average rating (C)
average_rating = statistics.mean(df_1['avgRating'])
print('The average rating across all movies is:', average_rating)

# Calculate the average number of votes (m)
avg_num_votes = statistics.mean(df_1['numVotes'])  # Use the average number of votes for threshold
print('The average number of votes is:', avg_num_votes)

# Create a new column 'score' for the weighted average rating using 'avgRating' and 'numVotes'
df_1['score'] = df_1.apply(lambda row: calculate_weighted_score(row['avgRating'], row['numVotes'], average_rating, avg_num_votes), axis=1)

# Display the DataFrame with the calculated weighted score
df_1[['movieId', 'title', 'avgRating', 'numVotes', 'score']].head()

The average rating across all movies is: 3.7323364168313313
The average number of votes is: 2.3743169398907105


,movieId,title,avgRating,numVotes,score
0,1,Toy Story (1995),3.833333,3,3.788714
3,2,Jumanji (1995),3.750000,4,3.743421
7,3,Grumpier Old Men (1995),2.500000,2,3.168895
9,5,Father of the Bride Part II (1995),1.500000,2,2.711680
11,7,Sabrina (1995),3.000000,1,3.515304


In [24]:
df_1.head()

,movieId,title,genres,userId,rating,tag,numVotes,avgRating,score
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,336,4.0,pixar,3,3.833333,3.788714
3,2,Jumanji (1995),Adventure|Children|Fantasy,62,4.0,fantasy,4,3.750000,3.743421
7,3,Grumpier Old Men (1995),Comedy|Romance,289,2.5,moldy,2,2.500000,3.168895
9,5,Father of the Bride Part II (1995),Comedy,474,1.5,pregnancy,2,1.500000,2.711680
11,7,Sabrina (1995),Comedy|Romance,474,3.0,remake,1,3.000000,3.515304


### <a id='toc1_5_1_'></a>[Exercise 1 - Get the top 5 suggestions sorting by score in descending order](#toc0_)


In [25]:
# TODO: filtering out the top 5 suggestions
# You can use `sort_values` to sort the DataFrame by the 'score' column in descending order

df_1.sort_values(by='score', ascending=False).head()

,movieId,title,genres,userId,rating,tag,numVotes,avgRating,score
199,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,103,5.0,good dialogue,181,4.983425,4.967226
1337,2959,Fight Club (1999),Action|Crime|Drama|Thriller,424,4.5,dark comedy,54,4.944444,4.893394
604,924,2001: A Space Odyssey (1968),Adventure|Drama|Sci-Fi,474,4.0,Hal,41,4.951220,4.884498
998,1732,"Big Lebowski, The (1998)",Comedy|Crime,474,3.5,Coen Brothers,32,4.953125,4.868802
164,293,Léon: The Professional (a.k.a. The Professiona...,Action|Crime|Drama|Thriller,166,4.5,assassin,35,4.928571,4.852577


<details>
    <summary>Click here for the solution</summary>

```python
# filtering out the top 5 suggestions
top_5_movies = df_1.sort_values(by = 'score', ascending = False).head(5)[['title', 'genres', 'tag', 'score']]
print('Top 5 movies:')
top_5_movies
```

</details>


## <a id='toc1_6_'></a>[Content-based recommendation](#toc0_)

Content-based filtering focuses on the attributes of items and the user's profile. It recommends movies to users based on features that closely match the user's profile. Movie A could be recommended because it matches the user's preferred genre, cast, and keywords. However, we might get limited diversity as it may not recommend items outside the user's known preferences, potentially limiting discovery of new types of items.

We want to compute the cosine similarity based on a number of features. Next, we will be creating a column `features` to gather the columns that we want to recommend to users. Calculation will be based on the type, genres, origin country, language, plot, summary, and cast.


In [26]:
# We will now create a new DataFrame that contains only the columns we need for our analysis.
df_2 = df_1[['movieId', 'title', 'userId', 'avgRating', 'numVotes', 'score', 'genres', 'tag']].copy()
df_2.reset_index(drop=True, inplace=True)
df_2.head()

,movieId,title,userId,avgRating,numVotes,score,genres,tag
0,1,Toy Story (1995),336,3.833333,3,3.788714,Adventure|Animation|Children|Comedy|Fantasy,pixar
1,2,Jumanji (1995),62,3.750000,4,3.743421,Adventure|Children|Fantasy,fantasy
2,3,Grumpier Old Men (1995),289,2.500000,2,3.168895,Comedy|Romance,moldy
3,5,Father of the Bride Part II (1995),474,1.500000,2,2.711680,Comedy,pregnancy
4,7,Sabrina (1995),474,3.000000,1,3.515304,Comedy|Romance,remake


In [27]:
# Replace '|' with spaces in 'genres' and combine it with 'tag' using a space
df_2['features'] = df_2['genres'].str.replace('|', ' ') + ' ' + df_2['tag'].fillna('')

df_2

,movieId,title,userId,avgRating,numVotes,score,genres,tag,features
0,1,Toy Story (1995),336,3.833333,3,3.788714,Adventure|Animation|Children|Comedy|Fantasy,pixar,Adventure Animation Children Comedy Fantasy pixar
1,2,Jumanji (1995),62,3.750000,4,3.743421,Adventure|Children|Fantasy,fantasy,Adventure Children Fantasy fantasy
2,3,Grumpier Old Men (1995),289,2.500000,2,3.168895,Comedy|Romance,moldy,Comedy Romance moldy
3,5,Father of the Bride Part II (1995),474,1.500000,2,2.711680,Comedy,pregnancy,Comedy pregnancy
4,7,Sabrina (1995),474,3.000000,1,3.515304,Comedy|Romance,remake,Comedy Romance remake
...,...,...,...,...,...,...,...,...,...
1459,183611,Game Night (2018),62,4.000000,3,3.881749,Action|Comedy|Crime|Horror,Comedy,Action Comedy Crime Horror Comedy
1460,184471,Tomb Raider (2018),62,3.500000,3,3.602644,Action|Adventure|Fantasy,adventure,Action Adventure Fantasy adventure
1461,187593,Deadpool 2 (2018),62,4.000000,3,3.881749,Action|Comedy|Sci-Fi,Josh Brolin,Action Comedy Sci-Fi Josh Brolin
1462,187595,Solo: A Star Wars Story (2018),62,4.000000,2,3.854716,Action|Adventure|Children|Sci-Fi,Emilia Clarke,Action Adventure Children Sci-Fi Emilia Clarke


Next, let's vectorize the features column using TF-IDF vectorizer. 
The Term Frequency-Inverse Document Frequency(TF-IDF) vectorizer is used to transform text into numerical representations. It evaluates the importance of a word in a document relative to a collection of documents by considering both its frequency within a specific document (TF) and its rarity across all documents (IDF).


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')

# Fit and transform the 'features' column to create TF-IDF vectors
X = vectorizer.fit_transform(df_2['features'])

In [29]:
X[:10]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 34 stored elements and shape (10, 852)>

Finally, let's get the cosine similarity and recommend items based on users' needs.


In [30]:
X.shape

(1464, 852)

In [31]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate Cosine Similarity
similarity = cosine_similarity(X)

In [32]:
similarity.shape

(1464, 1464)

In [33]:
similarity[:5]

array([[1.        , 0.54178665, 0.05126977, ..., 0.03469132, 0.16817315,
        0.20288257],
       [0.54178665, 1.        , 0.        , ..., 0.        , 0.20298276,
        0.        ],
       [0.05126977, 0.        , 1.        , ..., 0.04319465, 0.        ,
        0.05744448],
       [0.0620546 , 0.        , 0.07726505, ..., 0.05228084, 0.        ,
        0.0695282 ],
       [0.05831102, 0.        , 0.1754891 , ..., 0.04912688, 0.        ,
        0.06533375]])

In [34]:
# Recommendation function (including itself as first result)
def recommendation(title, df, similarity, top_n=3):
    try:
        # Get the index of the movie that matches the title
        idx = df[df['title'] == title].index[0]
    except IndexError:
        print(f"Movie '{title}' not found in the dataset.")
        return

    # Get the similarity scores for the given movie
    sim_scores = list(enumerate(similarity[idx]))

    # Sort the movies based on similarity scores in descending order
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Print the top_n most similar movies (including itself)
    print(f"Movies similar to '{title}' (First movie is itself):")
    for i, (index, score) in enumerate(sim_scores[:top_n+1]):
        movie = df.iloc[index]
        print(f"{i}. {movie['title']} (Similarity Score: {score:.3f})")
        print(f"   Genres: {movie['genres']}")
        print(f"   Tag: {movie['tag']}\n")

# Test the recommendation function
recommendation("Toy Story (1995)", df_2, similarity)

Movies similar to 'Toy Story (1995)' (First movie is itself):
0. Toy Story (1995) (Similarity Score: 1.000)
   Genres: Adventure|Animation|Children|Comedy|Fantasy
   Tag: pixar

1. Bug's Life, A (1998) (Similarity Score: 0.939)
   Genres: Adventure|Animation|Children|Comedy
   Tag: Pixar

2. Toy Story 2 (1999) (Similarity Score: 0.675)
   Genres: Adventure|Animation|Children|Comedy|Fantasy
   Tag: animation

3. Sintel (2010) (Similarity Score: 0.583)
   Genres: Animation|Fantasy
   Tag: adventure



### <a id='toc1_6_1_'></a>[Exercise 2 - Check the recommendations for the movie 'Toy Story 2 (1999)'](#toc0_)


In [35]:
# TODO
recommendation("Toy Story 2 (1999)", df_2, similarity)

Movies similar to 'Toy Story 2 (1999)' (First movie is itself):
0. Toy Story 2 (1999) (Similarity Score: 1.000)
   Genres: Adventure|Animation|Children|Comedy|Fantasy
   Tag: animation

1. Croods, The (2013) (Similarity Score: 0.856)
   Genres: Adventure|Animation|Comedy
   Tag: animation

2. Sintel (2010) (Similarity Score: 0.853)
   Genres: Animation|Fantasy
   Tag: adventure

3. Invincible Iron Man, The (2007) (Similarity Score: 0.775)
   Genres: Animation
   Tag: animation



<details>
    <summary>Click here for the solution</summary>

```python
recommendation("Toy Story 2 (1999)", df_2, similarity)
```

</details>


---


## <a id='toc1_7_'></a>[Collaborative filtering](#toc0_)

Collaborative filtering is a recommendation system technique that makes automatic predictions about a user’s preferences by collecting taste or preference information from many users. The assumption behind collaborative filtering is that if users agreed on certain items in the past, they are likely to agree on similar items in the future.

There are two primary approaches to collaborative filtering:

1.	User-based Collaborative Filtering: This method identifies users with similar preferences and recommends items that similar users have liked. In other words, a user receives recommendations based on the preferences of users who have historically rated items similarly.
2.	Item-based Collaborative Filtering: In this method, items similar to those the user has liked or rated highly in the past are recommended. The system identifies items that are frequently rated similarly across a user base and suggests items that share these patterns.



In [36]:
rating_df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [37]:
# Pivot user-item matrix from ratings
user_rating_matrix = rating_df.pivot(index="movieId", columns="userId", values="rating")

In [38]:
user_rating_matrix

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,NaN,NaN,4.0,NaN,4.5,NaN,NaN,NaN,...,4.0,NaN,4.0,3.0,4.0,2.5,4.0,2.5,3.0,5.0
2,NaN,NaN,NaN,NaN,NaN,4.0,NaN,4.0,NaN,NaN,...,NaN,4.0,NaN,5.0,3.5,NaN,NaN,2.0,NaN,NaN
3,4.0,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193585,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
# fill na with 0
user_rating_matrix = user_rating_matrix.fillna(0)

user_rating_matrix.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,0.0,0.0,4.0,0.0,4.5,0.0,0.0,0.0,...,4.0,0.0,4.0,3.0,4.0,2.5,4.0,2.5,3.0,5.0
2,0.0,0.0,0.0,0.0,0.0,4.0,0.0,4.0,0.0,0.0,...,0.0,4.0,0.0,5.0,3.5,0.0,0.0,2.0,0.0,0.0
3,4.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0


In this section, we will be using a NearestNeighbors classifier and using it based on the cosine similarity metric.


In [40]:
from sklearn.neighbors import NearestNeighbors

rec = NearestNeighbors(metric = 'cosine')
rec.fit(user_rating_matrix)

NearestNeighbors(metric='cosine')

Finally, here is our function to get 5 recommended items based on a movie previously watched.


In [41]:
def get_movie_id_from_title(title, df_movies):
    """
    Given a movie title, safely finds its corresponding movieId.
    Returns None if the movie doesn't exist.
    """
    movie_matches = df_movies[df_movies['title'] == title]
    if movie_matches.empty:
        print(f"Movie '{title}' not found in dataset.")
        return None
    return int(movie_matches.iloc[0]['movieId'])

def find_similar_movie_indices(movie_id, user_rating_matrix, knn_model, n_neighbors=15):
    """
    Takes a movieId and a trained KNN model, and returns a list of 
    movieIds that are most similar (excluding the queried movie itself).
    """
    try:
        # Find the row number for this movie in our matrix
        user_index = user_rating_matrix.index.get_loc(movie_id)
    except KeyError:
        print(f"Movie ID {movie_id} not found in the user rating matrix.")
        return None
        
    # Get the raw rating data for this movie and reshape for sklearn
    movie_ratings_row = user_rating_matrix.iloc[user_index].values.reshape(1, -1)
    
    # Let the KNN model find the nearest neighbors
    distances, indices = knn_model.kneighbors(movie_ratings_row, n_neighbors=n_neighbors)
    
    # Indices[0] contains the row numbers. 
    # We skip [0] because the closest neighbor to a movie is always itself!
    similar_row_indices = indices[0][1:]
    
    # Convert those row numbers back into actual movieIds
    similar_movie_ids = user_rating_matrix.iloc[similar_row_indices].index.tolist()
    return similar_movie_ids

def display_recommendations(title, df_movies, user_rating_matrix, knn_model):
    """
    The main orchestrator function. Gets the ID, finds neighbors, and formats the output.
    """
    movie_id = get_movie_id_from_title(title, df_movies)
    if not movie_id:
        return
        
    similar_ids = find_similar_movie_indices(movie_id, user_rating_matrix, knn_model)
    if not similar_ids:
        return
        
    # Filter the original dataframe to just those recommended IDs
    recommendations_df = df_movies[df_movies['movieId'].isin(similar_ids)]
    
    # Return just the top 5, showing only the relevant columns
    return recommendations_df[['title', 'avgRating', 'genres']].head(5)

In [42]:
display_recommendations('Toy Story (1995)', df_2, user_rating_matrix, rec)

,title,avgRating,genres
41,Apollo 13 (1995),4.500000,Adventure|Drama|IMAX
63,Star Wars: Episode IV - A New Hope (1977),4.527778,Action|Adventure|Sci-Fi
72,Pulp Fiction (1994),4.983425,Comedy|Crime|Drama|Thriller
88,Forrest Gump (1994),3.666667,Comedy|Drama|Romance|War
91,"Lion King, The (1994)",4.800000,Adventure|Animation|Children|Drama|Musical|IMAX


In [43]:
# # Function to get movie recommendations based on a title
# def get_recommendations(title):
#     # Get movie details
#     movie = df_2[df_2['title'] == title]
    
#     if movie.empty:
#         print(f"Movie '{title}' not found in dataset.")
#         return None
    
#     movie_id = int(movie['movieId'])
    
#     # Get the index of the movie in the user-item matrix
#     try:
#         user_index = user_rating_matrix.index.get_loc(movie_id)
#     except KeyError:
#         print(f"Movie ID {movie_id} not found in the user rating matrix.")
#         return None
    
#     # Get the user ratings for the movie
#     user_ratings = user_rating_matrix.iloc[user_index]
    
#     # Reshape the ratings to be a single sample (1, -1)
#     reshaped_df = user_ratings.values.reshape(1, -1)
    
#     # Find the nearest neighbors (similar movies)
#     distances, indices = rec.kneighbors(reshaped_df, n_neighbors=15)
    
#     # Get the movieIds of the nearest neighbors (excluding the first, which is the queried movie itself)
#     nearest_idx = user_rating_matrix.iloc[indices[0]].index[1:]
    
#     # Get the movie details for the nearest neighbors
#     nearest_neighbors = pd.DataFrame({'movieId': nearest_idx})
#     result = pd.merge(nearest_neighbors, df_2, on='movieId', how='left')
    
#     # Return the top recommendations
#     return result[['title', 'avgRating', 'genres']].head()

# # Test the recommendation function
# get_recommendations('Toy Story (1995)')

### <a id='toc1_7_1_'></a>[Exercise 3 - Check the recommendations for the movie 'Jurassic Park (1993)'](#toc0_)


In [44]:
display_recommendations('Jurassic Park (1993)', df_2, user_rating_matrix, rec)

,title,avgRating,genres
35,Braveheart (1995),4.350000,Action|Drama|War
41,Apollo 13 (1995),4.500000,Adventure|Drama|IMAX
88,Forrest Gump (1994),3.666667,Comedy|Drama|Romance|War
91,"Lion King, The (1994)",4.800000,Adventure|Animation|Children|Drama|Musical|IMAX
93,Speed (1994),4.000000,Action|Romance|Thriller


<details>
    <summary>Click here for the solution</summary>

```python
get_recommendations('Jurassic Park (1993)')
```

</details>


---


## <a id='toc1_8_'></a>[Authors](#toc0_)


[Lucy Xu](https://author.skills.network/instructors/lucy_xu)

[Ricky Shi](https://author.skills.network/instructors/ricky_shi)

## <a id='toc1_9_'></a>[Contributors](#toc0_)

[Hailey Quach](https://www.haileyq.com/)

Copyright © 2024 IBM Corporation. All rights reserved.
